<a href="https://colab.research.google.com/github/LukeRDuob/AI-Labsheets/blob/main/Lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This worksheet demonstrates the process of performing **linear asymmetric quantisation** on a floating-point matrix we looked at in Efficient AI. Similar to last week, you will do some work implementing your own version of the algorithm, to ensure that you understand the details.



1. Import key packages: torch, and any others that you prefer to work with. In general, when writing code, you will put all your import statements at the top. However, for these worksheets we will import as we go along.

In [13]:
import torch as t



2. Create a random 4×4 Floating-Point Matrix, which we will quantised next.

In [14]:
x = t.rand((4,4))
print(x)

tensor([[0.9718, 0.4618, 0.9095, 0.0107],
        [0.6950, 0.6759, 0.3178, 0.5020],
        [0.2093, 0.5500, 0.3103, 0.9042],
        [0.6553, 0.3251, 0.7717, 0.8170]])


3. Compute Quantisation Parameters (Scale & Zero-point) of this tensor. $ Scale = (x_{\text{max}} - x_{\text{min}}) / (q_{\text{max}} - q_{\text{min}}) $, $ Zero-point =q_{\text{min}} - round(\frac{x_{\text{min}}}{s}) $

In [15]:
# int8 integer range
q_min, q_max = -128, 127

# get data min/max
x_min = t.min(x)
x_max = t.max(x)
# compute scale and zero-point
scale = (x_max - x_min)/(q_max - q_min)
zero_point = q_min - t.round(x_min/scale)


# clamp zero_point to valid int8 range
zero_point = t.clip(zero_point,q_min, q_max)

4. Implement the quantisation formula:
$q = round(x / scale + zero\_point)$

In [16]:
def quantise(x, scale, zero_point,q_min,q_max):
  # Implement the quantization formula:
  q = t.round(x / scale)+ zero_point

  # Clamp to valid range:[q_min,q_max]
  q = t.clip(q, q_min, q_max)

  return q.to(t.int8)

# quantisation
x_q = quantise(x, scale, zero_point, q_min, q_max)
print("quantised matrix:\n",x_q)

quantised matrix:
 tensor([[ 127,   -5,  113, -125],
        [  56,   51,  -44,    5],
        [ -72,   18,  -46,  112],
        [  46,  -42,   77,   89]], dtype=torch.int8)


5. Implement dequantisation formula: $ dequant= (q - zero\_point) * scale $

In [17]:
def dequantise(q, scale, zero_point):
  # Convert q back to float and apply inverse mapping:
  x_dq = (q - zero_point) * scale
  return x_dq

x_dq = dequantise(x_q, scale, zero_point)
print("dequantised matrix:\n", x_dq)


dequantised matrix:
 tensor([[0.9610, 0.4636, 0.9083, 0.0113],
        [0.6935, 0.6746, 0.3166, 0.5013],
        [0.2111, 0.5502, 0.3090, 0.9045],
        [0.6558, 0.3241, 0.7726, 0.8178]])
